# LSTM-Autoencoder Feature Selection Comparison

This notebook implements a memory-efficient pipeline to compare three feature selection algorithms for anomaly detection using the CSE-CICIDS2018 dataset.

**Key Implementation Details:**
1. **Iterative Loading**: Processing data file-by-file to handle 16M+ rows without OOM.
2. **Temporal Splitting**: Splitting Benign data sequentially (no shuffle) to preserve LSTM dependencies.
3. **Benign Split Logic**: 
   - If Benign > 1M: Train = first 500k, Test = rest.
   - If Benign $\le$ 1M: 50:50 split (Test $\ge$ Train).
4. **Memory Optimization**: Downcasting to `float32` and using `HistGradientBoostingClassifier`.
5. **Threshold**: Set as $\max(\text{Training Reconstruction Error})$.

In [1]:
import os
import glob
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, precision_recall_curve, auc
)
import matplotlib.pyplot as plt
import time
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# --- Configuration ---
DATASET_DIR = "/home/dan/AnomalyIDS/dataset"
TIME_STEPS = 10
HIDDEN_DIM = 16
LEARNING_RATE = 0.001
DROPOUT = 0.2
BATCH_SIZE = 64
EPOCHS = 30
K_FEATURES = 10  # Number of features to select
BENIGN_TRAIN_CAP = 500000
BENIGN_THRESHOLD_LIMIT = 1000000

In [3]:
def optimize_df(df):
    """Downcast numeric columns to reduce memory usage."""
    float_cols = df.select_dtypes(include=['float']).columns
    int_cols = df.select_dtypes(include=['int']).columns
    df[float_cols] = df[float_cols].astype('float32')
    df[int_cols] = df[int_cols].astype('int32')
    return df

def process_file_splits(file_path):
    """Read a file and apply the specific Benign split logic."""
    df = pd.read_csv(file_path, low_memory=False)
    
    # Basic Cleaning
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(subset=numeric_cols + ['Label'], inplace=True)
    df = optimize_df(df)
    
    # Separate Benign and Attack
    benign = df[df['Label'] == 'Benign'].copy()
    attack = df[df['Label'] != 'Benign'].copy()
    
    # Benign Split Logic
    total_benign = len(benign)
    if total_benign > BENIGN_THRESHOLD_LIMIT:
        train_benign = benign.iloc[:BENIGN_TRAIN_CAP]
        test_benign = benign.iloc[BENIGN_TRAIN_CAP:]
    else:
        split_idx = total_benign // 2
        train_benign = benign.iloc[:split_idx]
        test_benign = benign.iloc[split_idx:]
    
    # Test set = test_benign + all attacks
    test_data = pd.concat([test_benign, attack], ignore_index=True)
    
    return train_benign, test_data

In [4]:
def get_representative_sample(dataset_dir, sample_size_per_class=100000):
    """Gather a balanced sample from all files for feature selection."""
    all_benign = []
    all_attack = []
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "*.csv")))
    
    for file in csv_files:
        df = pd.read_csv(file, low_memory=False)
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(subset=numeric_cols + ['Label'], inplace=True)
        
        benign = df[df['Label'] == 'Benign']
        attack = df[df['Label'] != 'Benign']
        
        if len(benign) > 0:
            all_benign.append(benign.sample(min(len(benign), sample_size_per_class // len(csv_files)), random_state=SEED))
        if len(attack) > 0:
            all_attack.append(attack.sample(min(len(attack), sample_size_per_class // len(csv_files)), random_state=SEED))
    
    df_benign = pd.concat(all_benign, ignore_index=True)
    df_attack = pd.concat(all_attack, ignore_index=True)
    
    combined = pd.concat([df_benign, df_attack], ignore_index=True)
    X = combined.drop(columns=['Label'])
    y = (combined['Label'] != 'Benign').astype(int).values
    
    # Final clean to ensure no NaNs for selection
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
    return X, y

In [5]:
def select_features_mi(X, y, k):
    mi_scores = mutual_info_classif(X, y, random_state=SEED)
    top_k_idx = np.argsort(mi_scores)[-k:]
    return X.columns[top_k_idx].tolist()

def select_features_rf(X, y, k):
    rf = HistGradientBoostingClassifier(random_state=SEED)
    rf.fit(X, y)
    # HistGradientBoosting does not have feature_importances_ like RF
    # We use a standard RF on the sample for the importance list
    from sklearn.ensemble import RandomForestClassifier
    rf_std = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rf_std.fit(X, y)
    importances = rf_std.feature_importances_
    top_k_idx = np.argsort(importances)[-k:]
    return X.columns[top_k_idx].tolist()

def select_features_rfe(X, y, k):
    from sklearn.ensemble import RandomForestClassifier
    rf = RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)
    rfe = RFE(estimator=rf, n_features_to_select=k, step=1)
    rfe.fit(X, y)
    return X.columns[rfe.support_].tolist()

methods = {
    "Mutual Information": select_features_mi,
    "RF Importance": select_features_rf,
    "RFE": select_features_rfe
}

In [6]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, num_features, time_steps, hidden_dim=16):
        super(LSTMAutoencoder, self).__init__()
        self.time_steps = time_steps
        self.num_features = num_features
        
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.encoder_dropout = nn.Dropout(0.2)
        
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.decoder_dropout = nn.Dropout(0.2)
        
        self.time_distributed = nn.Linear(hidden_dim, num_features)

    def forward(self, x):
        _, (hn, _) = self.encoder_lstm(x)
        latent = self.encoder_dropout(hn[-1])
        x_repeated = latent.unsqueeze(1).repeat(1, self.time_steps, 1)
        decoder_out, _ = self.decoder_lstm(x_repeated)
        decoder_out = self.decoder_dropout(decoder_out)
        reconstructed = self.time_distributed(decoder_out)
        return reconstructed

In [7]:
def create_sequences(data, seq_len):
    sequences = []
    for i in range(0, len(data) - seq_len + 1, seq_len):
        sequences.append(data[i : i + seq_len])
    return np.array(sequences)

def create_sequences_with_labels(data, labels, seq_len):
    sequences = []
    seq_labels = []
    for i in range(0, len(data) - seq_len + 1, seq_len):
        sequences.append(data[i : i + seq_len])
        seq_labels.append(1 if labels[i : i + seq_len].sum() > 0 else 0)
    return np.array(sequences), np.array(seq_labels)

In [8]:
def train_and_evaluate(method_name, selected_features, dataset_dir):
    print(f"\n>>> Starting pipeline for: {method_name}")
    
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "*.csv")))
    
    # --- PHASE 1: ITERATIVE TRAINING ---
    model = LSTMAutoencoder(len(selected_features), TIME_STEPS, HIDDEN_DIM).to(device)
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # To ensure consistent scaling, we fit scaler on a sample or first file
    # For simplicity and correctness, we use a global scaler fit on a sample of benigns
    scaler = StandardScaler()
    # Fit on first file's benign training set for simplicity
    tmp_train, _ = process_file_splits(csv_files[0])
    scaler.fit(tmp_train[selected_features].values)
    
    print("Training model iteratively...")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0
        total_samples = 0
        
        for file in csv_files:
            train_benign, _ = process_file_splits(file)
            
            # Scale and Sequence
            scaled = scaler.transform(train_benign[selected_features].values).astype(np.float32)
            seqs = create_sequences(scaled, TIME_STEPS)
            
            if len(seqs) == 0: continue
            
            tensor_x = torch.from_numpy(seqs).to(device)
            loader = DataLoader(TensorDataset(tensor_x, tensor_x), batch_size=BATCH_SIZE, shuffle=True)
            
            for bx, by in loader:
                optimizer.zero_grad()
                recon = model(bx)
                loss = criterion(recon, by)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * bx.size(0)
                total_samples += bx.size(0)
            
            del train_benign
            gc.collect()
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{EPOCHS} | Loss: {epoch_loss/total_samples:.6f}")
    
    # --- PHASE 2: THRESHOLD CALCULATION ---
    model.eval()
    train_errors = []
    with torch.no_grad():
        for file in csv_files:
            train_benign, _ = process_file_splits(file)
            scaled = scaler.transform(train_benign[selected_features].values).astype(np.float32)
            seqs = create_sequences(scaled, TIME_STEPS)
            if len(seqs) == 0: continue
            
            tensor_x = torch.from_numpy(seqs).to(device)
            recon = model(tensor_x)
            mse = ((tensor_x - recon)**2).mean(dim=(1, 2))
            train_errors.append(mse.cpu().numpy())
            del train_benign
            gc.collect()
    
    threshold = np.max(np.concatenate(train_errors))
    print(f"Max Train Error Threshold: {threshold:.6f}")
    
    # --- PHASE 3: ITERATIVE TESTING ---
    test_errors = []
    test_labels = []
    
    with torch.no_grad():
        for file in csv_files:
            _, test_df = process_file_splits(file)
            labels = (test_df['Label'] != 'Benign').astype(int).values
            scaled = scaler.transform(test_df[selected_features].values).astype(np.float32)
            
            seqs, seq_labels = create_sequences_with_labels(scaled, labels, TIME_STEPS)
            if len(seqs) == 0: continue
            
            tensor_x = torch.from_numpy(seqs).to(device)
            recon = model(tensor_x)
            mse = ((tensor_x - recon)**2).mean(dim=(1, 2))
            test_errors.append(mse.cpu().numpy())
            test_labels.append(seq_labels)
            
            del test_df
            gc.collect()
    
    test_errors = np.concatenate(test_errors)
    test_labels = np.concatenate(test_labels)
    y_pred = (test_errors > threshold).astype(int)
    
    # Metrics
    metrics = {
        "Selected Features": ", ".join(selected_features),
        "Accuracy": accuracy_score(test_labels, y_pred),
        "Precision": precision_score(test_labels, y_pred, zero_division=0),
        "Recall": recall_score(test_labels, y_pred, zero_division=0),
        "F1": f1_score(test_labels, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(test_labels, test_errors),
    }
    prec, rec, _ = precision_recall_curve(test_labels, test_errors)
    metrics["PR-AUC"] = auc(rec, prec)
    
    return metrics

In [ ]:
# --- EXECUTION LOOP ---
X_fs, y_fs = get_representative_sample(DATASET_DIR)
final_results = {}

for name, func in methods.items():
    selected_features = func(X_fs, y_fs, K_FEATURES)
    metrics = train_and_evaluate(name, selected_features, DATASET_DIR)
    final_results[name] = metrics

results_df = pd.DataFrame(final_results).T
print("\n=== FINAL COMPARISON ===")
print(results_df)

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
results_df[metrics_to_plot].plot(kind='bar', figsize=(14, 7))
plt.title("LSTM-Autoencoder Performance: Comparison of Feature Selection Methods")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()